In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from datetime import datetime
import glob
from reprojection import *
from show import *

In [2]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos_/*')))
#files = np.array(sorted(glob.glob('C:/Users/ikiru/data/4science/blos/*')))
#files = np.array(sorted(glob.glob('/home/ulyanov/data/sdo/hmi/2315/*')))
dates = np.array([datetime.fromisoformat(file.split('/')[-1].split('_')[-3]) for file in files])

#t = np.where(np.all([dates > datetime(2026,7,22),
#                     dates < datetime(2026,8,30)], axis=0))[0] # CR2314

#t = np.where(np.all([dates > datetime(2026,8,30),
#                     dates < datetime(2026,9,29)], axis=0))[0] # CR2315

t = np.where(np.all([dates > datetime(2026,8,25),
                     dates < datetime(2026,10,21)], axis=0))[0] # CR2315 +

files = files[t]
files[0], files[-1], len(files)

(np.str_('/home/ulyanov/data/solo/phi/2026/blos_/phi-fdt-blos_20260825T003003_V202610201946C_0648250501.fits'),
 np.str_('/home/ulyanov/data/solo/phi/2026/blos_/phi-fdt-blos_20261020T213003_V202610211240C_0650200508.fits'),
 221)

In [ ]:
data, view = make_map(files[:], kind='synoptic', a=12, b=0, c=0, car_rot=2.25, full_output=True)

In [ ]:
fig, ax = show_data(data[0], view, cmap='seismic', vmin=-20, vmax=20)

In [ ]:
plt.figure(figsize=(10,8))
plt.plot(np.nanstd(data[0], axis=1))
plt.plot(np.nanpercentile(data[0], 70, axis=1) - np.nanpercentile(data[0], 30, axis=1))

In [3]:
Q = []
a_range = np.arange(8, 16, 0.2)
for a in a_range:
    print(a)

    data, view = make_map(files[:], kind='synoptic', a=a, b=0, c=0, car_rot=2.25, full_output=True)
    Q += [data]

    np.savez('Q.npz', Q=np.array(Q))

Q = np.array(Q)

8.0


/home/ulyanov/PycharmProjects/reprojection/src/reprojection.py:107: RuntimeWarning: invalid value encountered in divide
  w = np.expand_dims(n / N, (-2, -1))
/home/ulyanov/PycharmProjects/reprojection/src/reprojection.py:144: RuntimeWarning: invalid value encountered in divide
  S = (FF - GF[:, :, -1:, :] ** 2 - Q ** 2 * (LL - GL[:, :, -1:, :] ** 2)) / LL / np.expand_dims(N - 1, (-1, -2))


8.2
8.399999999999999
8.599999999999998
8.799999999999997
8.999999999999996
9.199999999999996
9.399999999999995
9.599999999999994
9.799999999999994
9.999999999999993
10.199999999999992
10.399999999999991
10.59999999999999
10.79999999999999
10.99999999999999
11.199999999999989
11.399999999999988
11.599999999999987
11.799999999999986
11.999999999999986
12.199999999999985
12.399999999999984
12.599999999999984
12.799999999999983
12.999999999999982
13.199999999999982
13.39999999999998
13.59999999999998
13.79999999999998
13.999999999999979
14.199999999999978
14.399999999999977
14.599999999999977
14.799999999999976
14.999999999999975
15.199999999999974
15.399999999999974
15.599999999999973
15.799999999999972


In [43]:
#Q = np.load('Q.npz')['Q']

In [44]:
nx = Q.shape[-2]
lat = (np.arange(nx) / (nx - 1) * 2 - 1) * 90
#np.savez('diff_rot.npz', data=Q[:,0], a=a_range, lat=lat)

In [2]:
from utils import rebin

s = np.load('/home/ulyanov/data/4science/DR2315.npz')
Q = s['data']
a_range = s['a']
lat = s['lat']

Q = rebin(Q, 4, axis=1)
lat = lat[::4] + 1.5

In [3]:
Q.shape, lat.shape

((40, 180, 1800), (180,))

In [14]:
A, B, C = 14.712, -2.396, -1.787

#q = np.nanpercentile(Q, 90, axis=-1) - np.nanpercentile(Q, 10, axis=-1)
q = np.nanstd(Q, axis=-1)
q = np.nanargmax(np.nan_to_num(q), axis=0)

y = np.sin(lat * np.pi / 180)

plt.figure(figsize=(10,8))
plt.plot(lat, a_range[q], lw=2, label='FDT, CR2315')
plt.plot(lat, A + B * y ** 2 + C * y ** 4, '--', lw=2, label='Snodgrass, 1990')

plt.xlabel('Latitude, degrees')
plt.ylabel('Rotation rate, deg/day')

plt.legend()
plt.grid(True)
plt.xlim(0,90)
plt.ylim(8,16)
plt.tight_layout()

In [14]:
plt.figure(figsize=(10,8))
plt.imshow(Q[-10], origin='lower', aspect='auto', cmap='seismic', interpolation='none', vmin=-25, vmax=25)
plt.tight_layout()